<p align="center">
<img src="Images/sorbonne_logo.png" alt="Logo" width="300"/>
</p>

# **Module 2 - Statistics & Data Transformation**

* **Author**: Elia Landini
* **Student ID**: 12310239
* **Course**: EESM2-Financial Economics 
* **Supervisor**: Jean-Bernard Chatelain
* **Reference Repository**: https://github.com/EliaLand/Policy_Target_RegimeSwitchingPersistence

### **1) REQUIREMENTS SET-UP**

In [ ]:
# Requirements.txt file installation
# !pip install -r requirements.txt

In [ ]:
# Libraries import
import warnings
import pandas as pd
import numpy as np
import random
import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib.colors import Normalize
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
%matplotlib inline
import seaborn as sns
import scipy.stats as stats
from scipy.stats import norm
from scipy.stats import levene
from scipy.stats import ks_2samp
from scipy.stats import kstest
from scipy.stats import pearsonr
from scipy.stats import gaussian_kde
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.ar_model import AutoReg
from statsmodels.tsa.filters.hp_filter import hpfilter
import sklearn.tree
import sklearn.metrics
import sklearn.metrics
import sklearn.model_selection
import sklearn.preprocessing 
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (roc_auc_score, roc_curve, confusion_matrix,
                             precision_score, recall_score, f1_score,
                             accuracy_score, precision_recall_curve, auc, 
                             RocCurveDisplay, ConfusionMatrixDisplay)
from sklearn.linear_model import (LinearRegression, LogisticRegression)
from sklearn.calibration import calibration_curve, CalibratedClassifierCV
from sklearn.utils.class_weight import compute_class_weight
import plotly.express as px
import openpyxl as pxl
from stargazer.stargazer import Stargazer
from IPython.core.display import HTML
from IPython.display import Image
import itertools
from arch.unitroot import PhillipsPerron

### **2) DESCRIPTIVE STATISTICS (RAW)**

In [ ]:
# df import
jp_aggregated_df = pd.read_csv("Data/Aggregated/jp_aggregated_df.csv")
jp_aggregated_df

In [ ]:
# Metrics clusters for plotting
metrics_group = {
    "Monetary Aggregates" : ["Monetary Aggregates - M1 (JPY)", "Monetary Aggregates - M2 (JPY)", "Monetary Aggregates - M3 (JPY)"],
    "Credit Metrics" : ["Total Credit - General Government (%GDP)", "Total Credit - Households & NPISHs (%GDP)", "Total Credit - Private Non-Financial (%GDP)"],
    "Reserves" : ["Total Treasury Reserves (- Gold)"],
    "Monetary Policy Proxies (Yields)" : ["10-Year Gov Bond Yields (%)", "Call Money/Interbank Immediate (%)", "Est. 1-year Neutral Interest Rate (%)", "Est. 10-year Neutral Interest Rate (%)"], 
    "Exchange Rate" : ["USD-JPY reer CPI-based (Index 2015=100)", "JPY-USD Spot Exchange Rate"],
    "Output-Trends": ["Real GDP (billions chained 2015 JPY)"],
    "Consumption Prices": ["HICP (NSA)"],
    "Debt Metrics" : ["Central Government Debt (% GDP)", "Domestic Private Debt Securities (% GDP)", "Domestic Public Debt Securities (% GDP)"],
    "Banking Sector" : ["Loan Interest Rate (%)", "Deposit Interest Rate (%)", "1615.T-Price"],
    "Controls" : ["10-Year US T-Bills Yield (%)", "CBOE-VIX"],
    "BoJ Total Assets" : ["BoJ’s Total Assets (100 Million Yen)"]
}

In [ ]:
# Descriptive statistics summary table - aggregate data
df = jp_aggregated_df.copy()
df.describe()

### **3) NON-STATIONARITY CORRECTIONS**

In [ ]:
# Non-Stationarity Corrections - Log-Difference Transformations
# (!!!) Basically we need a detrending transformation for all the variables as expected, given the undisputable presence of unit-root root and so non-sttaionarity
# (!!!) Autocorrelation is also evident, suggesting a marked time-dependent component, and so a trend, so we'll opt for both log transformations as well as first differences 
df = jp_aggregated_df.copy()
trans_df = df[["Country", "Time"]].copy()

# Transformations: 
# - Monetary Aggregates: I(1) nominal levels (levels non-stationary, but first-differences are I(0), stationary)
# - Reserves: I(1), level series of policy shocks 
# - Exchange Rate: Log-difference (returns)
# - Consumption Prices: Log-difference (inflation)
# - Banking metruics: Log-difference (returns)
# - BoJ’s Total Assets: CA smoothing 
log_transformed_variables = ["Monetary Aggregates - M1 (JPY)", "Monetary Aggregates - M2 (JPY)", "Monetary Aggregates - M3 (JPY)",
                             "Total Treasury Reserves (- Gold)",
                             "USD-JPY reer CPI-based (Index 2015=100)", "JPY-USD Spot Exchange Rate",
                             "HICP (SA)",
                             "1615.T-Price",
                             "BoJ’s Total Assets (100 Million Yen)", 
                             "Call Money/Interbank Immediate (%)"
                            ]

# Log-Difference Transformed Variables and Annualization (percentage change over 12 months)
for var in log_transformed_variables:
    if var != "HICP (SA)":
        trans_df[f"LogDiff-{var}"] = np.log(df[f"{var}"]).diff()
    else:
        trans_df[f"LogDiff-{var}"] = np.log(df[f"{var}"]).diff()*12*100
trans_df


In [ ]:
# Non-Stationarity Corrections - AR(1) detrending 
df = jp_aggregated_df.copy()
trans_df = trans_df.copy()

# Transformations: 
# - Credit Metrics: AR(1) detrending (cyclical credit gap), we want to remove the persistence of credit t-1, we can reduce this relatiosnship to:
# (!!!) c_t = α + ρc_t−1​ + ε_t, we take the residuals 
# (!!!) first differences are to aggresive for credit metrics, they destroy medium-term cycles, signal-to-noise
# (!!!) credit metrics tend to be highly-persistent I(0) and near-unit-root stationary (so it might result non-stationary, but it is actually slowly mean reverting, we want to keep cyclical deviations)
# - Monetary Policy Proxies: AR(1) detrending (policy shocks), rates are persistent but not truly I(1)
# - Debt Metrics: same reasoning as for credit 
# - Banking metrics: same reasoning as for yields
# - VIX index
ar1detrend_transformed_variables = ["Total Credit - General Government (%GDP)", "Total Credit - Households & NPISHs (%GDP)", "Total Credit - Private Non-Financial (%GDP)",
                                    "10-Year Gov Bond Yields (%)", "Call Money/Interbank Immediate (%)", "Est. 1-year Neutral Interest Rate (%)", "Est. 10-year Neutral Interest Rate (%)",
                                    "Central Government Debt (% GDP)", "Domestic Private Debt Securities (% GDP)", "Domestic Public Debt Securities (% GDP)",
                                    "Loan Interest Rate (%)", "Deposit Interest Rate (%)",
                                    "10-Year US T-Bills Yield (%)", "CBOE-VIX"
                                   ]

# AR(1) detrending Transformed Variables 
for var in ar1detrend_transformed_variables:
# Lag-1 Reg
# (!!!) It triggers an errors as we don't specificy the model time index and frequency. It should be harmless, it only deactivates forecasting features.
    model = AutoReg(df[f"{var}"].dropna(), lags=1, old_names=False).fit()
# Residuals 
    trans_df[f"AR(1)detrend-{var}"] = df[f"{var}"] - model.fittedvalues
trans_df

In [ ]:
# Non-Stationarity Corrections - HP-filter cycle 
df = jp_aggregated_df.copy()
trans_df = trans_df.copy()

# Transformations: 
# - Output Trends: HP-filter cycle, trend smoothing
hpfilter_transformed_variables = ["Real GDP (billions chained 2015 JPY)"
                                 ]

# HP-filter cycle Transformed Variables 
for var in hpfilter_transformed_variables:    
    cycle, trend = hpfilter(
        np.log(df[f"{var}"]).dropna(),
        lamb=1600
    )
    trans_df[f"HPfilter-{var}"] = cycle
trans_df

In [ ]:
# Transformed df to csv
jp_trans_df = trans_df.copy()
jp_trans_df.to_csv("Data/Transformed/jp_trans_df.csv", index=False)

In [ ]:
# Core Transformed to csv 
jp_core_trans_df = jp_trans_df[["Country", "Time", "LogDiff-HICP (SA)", "AR(1)detrend-Call Money/Interbank Immediate (%)", "AR(1)detrend-10-Year Gov Bond Yields (%)"]].copy()
df = jp_core_trans_df.copy()
df.to_csv("Data/Transformed/jp_core_trans_df.csv", index=False)

In [ ]:
# Core Hybrid to csv

# Policy Target (transformed)
df1 = jp_core_trans_df.copy()
df1 = df1.reset_index()
df1 = df1.drop(columns=["AR(1)detrend-Call Money/Interbank Immediate (%)", "AR(1)detrend-10-Year Gov Bond Yields (%)"])

# Policy Instrument (raw)
df2 = jp_aggregated_df.copy()
df2 = df2.reset_index()
df2 = df2[["Country", "Time", "Call Money/Interbank Immediate (%)", "10-Year Gov Bond Yields (%)"]]

jp_core_hybrid_df = pd.merge(df1, df2, on=["Country", "Time"], how="outer")
jp_core_hybrid_df = jp_core_hybrid_df.drop(columns="index")
jp_core_hybrid_df = jp_core_hybrid_df.dropna()

jp_core_hybrid_df.to_csv("Data/Transformed/jp_core_hybrid_df.csv", index=False)
jp_core_hybrid_df

### **4) DESCRIPTIVE STATISTICS (TRANSFORMED)**

In [ ]:
# Descriptive statistics summary table - Raw Data
df = jp_aggregated_df[["Country", "Time", "HICP (SA)", "Call Money/Interbank Immediate (%)", "10-Year Gov Bond Yields (%)"]].copy()
df.describe()

In [ ]:
# Descriptive statistics summary table - Transformed Data
df = jp_core_trans_df.copy()
df.describe()